# Day 17: Complete Matrix Formalization of Financial Event Algebra

*Alpha Flow Research · HongJin HE · July 2026*

---

## The State Vector

Every asset $i$ in the universe is described by a $d=5$ dimensional state vector:

$$s^{(i)} = \begin{bmatrix} p^{(i)} \\ v^{(i)} \\ \ell^{(i)} \\ \kappa^{(i)} \\ \iota^{(i)} \end{bmatrix} \in \mathbb{R}^5$$

| Coordinate | Symbol | Meaning |
|---|---|---|
| 0 | $p^{(i)} = \log S^{(i)}$ | Log price |
| 1 | $v^{(i)} = \log V^{(i)}$ | Log trading volume |
| 2 | $\ell^{(i)} = \log(D/E)^{(i)}$ | Log leverage (debt/equity) |
| 3 | $\kappa^{(i)} = \log N^{(i)}$ | Log shares outstanding |
| 4 | $\iota^{(i)} \in [0,1]$ | Information disclosure level |

For $n$ assets: $s = [s^{(1)\top}, s^{(2)\top}, \ldots, s^{(n)\top}]^\top \in \mathbb{R}^{nd}$

## The Universal Operator Form

Every financial event $w$ maps to an **affine operator**:

$$\boxed{T_w(s) = A_w \cdot s + b_w + \Sigma_w \varepsilon_w, \quad \varepsilon_w \sim \mathcal{N}(0, I)}$$

- $A_w \in \mathbb{R}^{md \times nd}$: how existing states transform
- $b_w \in \mathbb{R}^{md}$: deterministic shift (known impact)
- $\Sigma_w \in \mathbb{R}^{md \times md}$: uncertainty (how precisely we know the impact)

For $m = n$: dimension-preserving (Modes I, II). For $m \neq n$: dimension-changing (Mode III).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import sys
sys.path.insert(0, '..')

from events.operators import (
    # Mode I
    stock_split_operator, reverse_split_operator, dividend_operator,
    secondary_offering_operator, share_buyback_operator,
    earnings_shock_operator, analyst_rating_operator,
    index_change_operator, trading_halt_operator, short_squeeze_operator,
    # Mode II
    rate_change_operator, qe_operator, qt_operator,
    systemic_crisis_operator, circuit_breaker_operator,
    volatility_regime_shift_operator, inflation_shock_operator,
    # Mode III
    merger_operator, spinoff_operator, ipo_operator,
    delisting_operator, bankruptcy_operator,
    # Composition
    compose, event_sequence,
    EventMode,
)

d = 5

# Print summary of all operators
print('=== COMPLETE EVENT OPERATOR CATALOGUE ===')
print()
print('MODE I — Local endomorphisms (single-asset, dimension-preserving)')
mode_i_ops = [
    ('Stock split 2:1',         stock_split_operator(2.0, n=1)),
    ('Reverse split 5:1',       reverse_split_operator(5.0, n=1)),
    ('Dividend 2%',             dividend_operator(0.02, n=1)),
    ('Secondary offering 10%',  secondary_offering_operator(0.10, n=1)),
    ('Buyback 5%',              share_buyback_operator(0.05, n=1)),
    ('Earnings shock +20%',     earnings_shock_operator(20.0, n=1)),
    ('Analyst upgrade',         analyst_rating_operator(+1, n=1)),
    ('Analyst downgrade',       analyst_rating_operator(-1, n=1)),
    ('Index inclusion',         index_change_operator(True, n=1)),
    ('Trading halt 2h',         trading_halt_operator(2.0, n=1)),
    ('Short squeeze 0.7',       short_squeeze_operator(0.7, n=1)),
]
for label, op in mode_i_ops:
    print(f'  {label:<30}: A {op.A_w.shape}, b_max={np.abs(op.b_w).max():.4f}, Σ_diag_max={np.diag(op.Sigma_w).max():.4f}')

print()
print('MODE II — Global tensor product (macro events, all assets simultaneously)')
n_test = 3
mode_ii_ops = [
    ('Rate hike +25bps',        rate_change_operator(+25, n=n_test)),
    ('Rate cut -50bps',         rate_change_operator(-50, n=n_test)),
    ('QE $500B',                qe_operator(500, n=n_test)),
    ('QT $300B',                qt_operator(300, n=n_test)),
    ('Systemic crisis (0.7)',   systemic_crisis_operator(0.7, n=n_test)),
    ('Circuit breaker',         circuit_breaker_operator(n=n_test)),
    ('Vol regime 15→30',        volatility_regime_shift_operator(30, 15, n=n_test)),
    ('CPI beat +0.5%',          inflation_shock_operator(0.5, n=n_test)),
]
for label, op in mode_ii_ops:
    print(f'  {label:<30}: A {op.A_w.shape}, b_max={np.abs(op.b_w).max():.4f}')

print()
print('MODE III — Pairwise morphisms (universe size changes)')
mode_iii_ops = [
    ('IPO NEWCO $90',           ipo_operator('NEWCO', 90.0, n=3)),
    ('Merger (0→1, 30% prem)',  merger_operator(0, 1, n=3)),
    ('Spin-off (30% carved)',   spinoff_operator(0, n=3)),
    ('Delisting asset 2',       delisting_operator(2, n=4)),
    ('Bankruptcy (10% recov)',  bankruptcy_operator(1, n=3, recovery_rate=0.10)),
]
for label, op in mode_iii_ops:
    n_in = op.source_dim
    n_out = op.target_dim
    print(f'  {label:<30}: A {op.A_w.shape}  [{n_in}→{n_out} assets]')

## Matrix Visualization: What Does A_w Look Like?

For each mode, the structure of $A_w$ tells us the **information flow** between asset state components.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

ops_to_viz = [
    ('Mode I: Stock Split 2:1\n(1 asset)', stock_split_operator(2.0, n=1)),
    ('Mode I: Earnings Shock +20%\n(2 assets)', earnings_shock_operator(20.0, asset_idx=0, n=2)),
    ('Mode II: Rate Hike 25bp\n(3 assets)', rate_change_operator(25, n=3)),
    ('Mode II: Systemic Crisis\n(3 assets)', systemic_crisis_operator(0.7, n=3)),
    ('Mode III: Merger (A→B, 3 assets)', merger_operator(0, 1, n=3)),
    ('Mode III: Spin-off (3 assets)', spinoff_operator(0, n=3)),
]

state_labels = ['p', 'v', 'ℓ', 'κ', 'ι']

for ax, (title, op) in zip(axes.flat, ops_to_viz):
    A = op.A_w
    n_src = op.source_dim
    n_tgt = op.target_dim
    
    im = ax.imshow(A, cmap='RdBu', vmin=-1.5, vmax=1.5, aspect='auto')
    ax.set_title(title, fontsize=9)
    
    # Label axes
    src_labels = [f'{s}_{i}' for i in range(n_src) for s in state_labels]
    tgt_labels = [f'{s}_{i}' for i in range(n_tgt) for s in state_labels]
    
    if len(src_labels) <= 15:
        ax.set_xticks(range(len(src_labels)))
        ax.set_xticklabels(src_labels, rotation=45, fontsize=7)
        ax.set_yticks(range(len(tgt_labels)))
        ax.set_yticklabels(tgt_labels, fontsize=7)
    
    ax.set_xlabel(f'Source ({n_src} assets)', fontsize=8)
    ax.set_ylabel(f'Target ({n_tgt} assets)', fontsize=8)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Event Operator Matrices A_w: Structure by Mode', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/operator_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Key observations:')
print('  Mode I:   A_w ≈ I (identity + small perturbation)')
print('  Mode II:  A_w = I (b_w does the work — global shift)')
print('  Mode III: A_w is non-square (dimension-changing linear map)')

## Groupoid Composition: Building Complex Events from Simple Ones

The groupoid structure means we can **compose events** to model realistic market sequences.

Example: *2024 scenario — Fed hikes → company can't refinance → goes bankrupt*

$$T_{\text{bankruptcy}} \circ T_{\text{rate hike}}$$

This is valid because $\dim(\text{target}(T_{\text{rate hike}})) = \dim(\text{source}(T_{\text{bankruptcy}}))$.

But:
$$T_{\text{merger}} \circ T_{\text{bankruptcy}}$$

requires $n-1 = $ source of merger's input universe — only valid if merger is for a post-bankruptcy universe.

In [ ]:
# Simulate a realistic market event timeline: 4 assets, 6 months of events
n_init = 4
d = 5

# Initial market state: 4 assets at 'normal' levels
s0 = np.zeros(n_init * d)
for i in range(n_init):
    s0[i * d + 0] = np.log(100 + i * 20)  # log prices: $100, $120, $140, $160
    s0[i * d + 1] = 4.0                    # log volume
    s0[i * d + 2] = -0.5                   # moderate leverage
    s0[i * d + 3] = np.log(1e9)            # $1B market cap (rough)
    s0[i * d + 4] = 0.6                    # moderate transparency

rng = np.random.default_rng(2026)

# Event sequence (realistic market scenario)
event_timeline = [
    # Month 1: Fed hikes 50bp
    ('Month 1: Fed +50bp',
     rate_change_operator(+50, n=4)),
    # Month 2: Asset 2 gets analyst downgrade
    ('Month 2: Asset 2 downgraded',
     analyst_rating_operator(-1, asset_idx=2, n=4)),
    # Month 3: Asset 3 goes bankrupt (universe: 4 → 3)
    ('Month 3: Asset 3 bankruptcy',
     bankruptcy_operator(asset_idx=3, n=4, recovery_rate=0.05)),
    # Month 4: Asset 0 splits 2:1 (universe still 3)
    ('Month 4: Asset 0 splits 2:1',
     stock_split_operator(2.0, asset_idx=0, n=3)),
    # Month 5: CPI beat
    ('Month 5: CPI +0.4% beat',
     inflation_shock_operator(0.4, n=3)),
    # Month 6: New IPO (universe: 3 → 4)
    ('Month 6: ARM-style IPO $60',
     ipo_operator('NEWCO', 60.0, n=3)),
]

print('Event Timeline Simulation')
print('=' * 70)
state = s0.copy()
n_current = n_init
state_history = [state.copy()]
n_history = [n_current]
labels = ['t=0 (initial)']

for event_label, op in event_timeline:
    n_in = op.source_dim if op.source_dim > 0 else len(state) // d
    actual_n = len(state) // d

    if n_in > 0 and n_in != actual_n:
        print(f'  SKIP {event_label}: expected {n_in} assets, have {actual_n}')
        continue

    state = op.apply(state, rng=rng)
    n_current = op.target_dim if op.target_dim > 0 else len(state) // d
    state_history.append(state.copy())
    n_history.append(n_current)
    labels.append(event_label)

    print(f'  {event_label}')
    print(f'    Universe: {n_in}→{n_current} assets')
    print(f'    Log prices: {[f"{state[i*d]:.3f}" for i in range(n_current)]}')

print('\nFinal universe size:', n_current, 'assets')
print('Initial universe size:', n_init, 'assets')

In [ ]:
# Visualize price trajectory through events
fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# Pad prices to common length for visualization (NaN for non-existent assets)
max_assets = max(n_history)
price_history = np.full((len(state_history), max_assets), np.nan)
for t, (state, n) in enumerate(zip(state_history, n_history)):
    for i in range(min(n, max_assets)):
        if i * d < len(state):
            price_history[t, i] = np.exp(state[i * d])  # log → price

colors_assets = ['steelblue', 'orange', 'green', 'red', 'purple']
for i in range(max_assets):
    valid = ~np.isnan(price_history[:, i])
    t_valid = np.where(valid)[0]
    if len(t_valid) > 0:
        axes[0].plot(t_valid, price_history[valid, i],
                    marker='o', ms=6, lw=2,
                    color=colors_assets[i % len(colors_assets)],
                    label=f'Asset {i}')

axes[0].set_ylabel('Price ($)')
axes[0].set_title('Asset Prices Through Event Sequence')
axes[0].set_xticks(range(len(labels)))
axes[0].set_xticklabels(labels, rotation=20, ha='right', fontsize=8)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(len(n_history)), n_history, color='teal', alpha=0.7)
axes[1].set_xticks(range(len(labels)))
axes[1].set_xticklabels(labels, rotation=20, ha='right', fontsize=8)
axes[1].set_ylabel('Universe size (# assets)')
axes[1].set_title('Universe Size: Mode III Events Change n')
axes[1].set_ylim(0, max_assets + 1)
axes[1].grid(True, alpha=0.3, axis='y')

for t, (n_prev, n_next) in enumerate(zip(n_history[:-1], n_history[1:])):
    if n_next != n_prev:
        axes[1].axvline(t + 0.5, color='red', ls='--', alpha=0.7)
        axes[1].text(t + 0.5, max_assets * 0.9,
                    f'Mode III\n{n_prev}→{n_next}',
                    ha='center', fontsize=8, color='red')

plt.tight_layout()
plt.savefig('../figures/event_sequence_simulation.png', dpi=150, bbox_inches='tight')
plt.show()

## Why This Matters: The Complete Algebra

### Theorem 7 (Groupoid Structure)

The collection $\mathcal{G} = \{T_w : w \in \mathcal{W}\}$ with composition $\circ$ forms a **groupoid** — specifically, a category where:

1. **Objects**: integers $\{n : n \ge 0\}$ (universe sizes)
2. **Morphisms**: $\text{Hom}(n, m) = \{T_w : \mathbb{R}^{nd} \to \mathbb{R}^{md}\}$
3. **Composition**: $T_1 \circ T_2 \in \text{Hom}(n, k)$ if $T_2 \in \text{Hom}(n, m)$, $T_1 \in \text{Hom}(m, k)$
4. **Identities**: $\text{Id}_n \in \text{Hom}(n, n)$ (zero event)

### Complete Event Taxonomy

| Event | Mode | $A_w$ structure | $b_w$ dominant terms |
|---|---|---|---|
| Stock split $k:1$ | I | $I_{nd}$ | $b_p = -\log k$, $b_\kappa = +\log k$ |
| Reverse split | I | $I_{nd}$ | $b_p = +\log k$, $b_\kappa = -\log k$ |
| Dividend $q$ | I | $I_{nd}$ | $b_p = -\log(1+q)$ |
| Secondary offering $f$ | I | $I_{nd}$ | $b_p < 0$, $b_\kappa > 0$, $b_\ell < 0$ |
| Buyback $f$ | I | $I_{nd}$ | $b_p > 0$, $b_\kappa < 0$, $b_\ell > 0$ |
| Earnings shock $\Delta$ | I | $I_{nd}$ | $b_p = 0.03\Delta$, $b_v > 0$, $b_\iota > 0$ |
| Analyst up/down | I | $I_{nd}$ | $b_p = \pm 0.03$, $b_v > 0$ |
| Index change | I | $I_{nd}$ | $b_p = \pm 0.035$, $b_v > 0$ |
| Trading halt | I | $I_{nd}$ | $b_v \to -\infty$, large $\Sigma_p$ |
| Short squeeze | I | $A_{pp} > 1$ | $b_p \gg 0$, $b_v \gg 0$ |
| Rate hike/cut | II | $I_{nd}$ | $b_p^{(i)} = -\text{dur}_i \cdot \Delta r$ |
| QE/QT | II | $I_{nd}$ | $b_p^{(i)} = \pm c$, $b_\ell < 0$ |
| Systemic crisis | II | $I_{nd}$ | $b_p \ll 0$, high correlated $\Sigma$ |
| Circuit breaker | II | $I_{nd}$ | $b_v \to -\infty$ (all assets) |
| CPI shock | II | $I_{nd}$ | $b_p = -0.005\cdot\text{surprise}$ |
| Merger | III | $(n-1)d \times nd$ | $b_p > 0$ (premium) |
| Spin-off | III | $(n+1)d \times nd$ | $b_p^{\text{parent}} < 0$, $b_p^{\text{child}} > 0$ |
| IPO | III | $(n+1)d \times nd$ | $b_{\text{new}} = \text{IPO price vector}$ |
| Delisting | III | $(n-1)d \times nd$ | $b = 0$ (clean removal) |
| Bankruptcy | III | $(n-1)d \times nd$ | $b_p^{\text{bankrupt}} = \log(\text{recovery})$ |

### Key Properties

1. **Modes I+II** operators form a **monoid** on $\mathbb{R}^{nd}$: closed under composition, have identity.
2. **Mode III** operators break closure: $T_{\text{merger}} \circ T_{\text{merger}}$ requires universe to have two pairs to merge.
3. **Groupoid** is the minimal structure containing all three modes.
4. **Short squeeze** is the only Mode I operator with $A_w \neq I$ — it introduces positive feedback in the $p$-$p$ entry. This is the mathematical fingerprint of momentum cascades.